In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder

# Laadi andmestik (asenda faili tee õigega)
df = pd.read_csv("https://raw.githubusercontent.com/matikukk/ti_kursus_data/main/Housing.csv")  # Asenda oma faili tee siia

# Vaata esimesi ridu
print(df.head())

# Kontrolli puuduvate väärtuste olemasolu
print(df.isnull().sum())

# Teisenda kategoorilised tunnused numbriliseks kujuks (kasutame LabelEncoder)
# Kasutame iga kategoorilise tunnuse jaoks eraldi LabelEncoderit
label_encoders = {} # Dictionary to store LabelEncoders for each column
for col in df.columns:
    if df[col].dtype == 'object':
        le = LabelEncoder() # Create a new LabelEncoder for each column
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le # Store the fitted LabelEncoder

# Jaga andmed treening- ja testandmeteks
X = df.drop('price', axis=1)  # Sõltuv muutuja on 'price'
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treeni lineaarne regressioon
lr = LinearRegression()
lr.fit(X_train, y_train)

# Treeni Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42) # Added random_state for reproducibility
rf.fit(X_train, y_train)

# Hindame mudeleid R² skoori abil
y_pred_lr = lr.predict(X_test)
r2_lr = r2_score(y_test, y_pred_lr)

y_pred_rf = rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Lineaarne regressioon: R² = {r2_lr}")
print(f"Random Forest: R² = {r2_rf}")

# Vali parim mudel
if r2_rf > r2_lr:
    parim_mudel = "Random Forest"
else:
    parim_mudel = "Lineaarne regressioon"

print(f"Parim mudel: {parim_mudel}")

# Ennusta hind uue maja jaoks (kasuta Random Foresti, kui see on parem)
new_data_dict = {
    'area': 3550,
    'bedrooms': 3,
    'bathrooms': 1,
    'stories': 1,
    'mainroad': label_encoders['mainroad'].transform(['yes'])[0], # Use specific encoder and get scalar
    'guestroom': label_encoders['guestroom'].transform(['no'])[0],
    'basement': label_encoders['basement'].transform(['no'])[0],
    'hotwaterheating': label_encoders['hotwaterheating'].transform(['no'])[0],
    'airconditioning': label_encoders['airconditioning'].transform(['yes'])[0],
    'parking': 1,
    'prefarea': label_encoders['prefarea'].transform(['yes'])[0],
    'furnishingstatus': label_encoders['furnishingstatus'].transform(['unfurnished'])[0]
}

# Create a DataFrame for prediction, ensuring column order matches training data
new_df_for_prediction = pd.DataFrame([new_data_dict], columns=X_train.columns)

if parim_mudel == "Random Forest":
    predicted_price = rf.predict(new_df_for_prediction)
else:
    predicted_price = lr.predict(new_df_for_prediction)

print(f"Ennustatud hind: {predicted_price[0]}")

      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  
price               0
area                0
bedrooms            0
bathrooms           0
stories    